# FlashInspector AI – Test Model on Video

Upload a video and run your trained YOLOv8 models on it.

Runs **both** models on every frame and combines the annotations:
- **Detection** (`best_detect.pt`) — bounding boxes for fire safety equipment (S1,2,5,6)
- **Segmentation** (`best_segment.pt`) — masks for extinguisher condition/occlusion (S3,4)

**Steps:** Check GPU → Install deps → Get code → Upload models → Upload video → Run inference → View & download results

## 1. Check GPU

Go to **Runtime → Change runtime type → T4 GPU** if no GPU is detected.

In [ ]:
!nvidia-smi 2>/dev/null || echo "\n\u26a0\ufe0f No GPU detected. Go to Runtime \u2192 Change runtime type \u2192 T4 GPU."

## 2. Install dependencies & clone repo

In [ ]:
!pip install -q ultralytics opencv-python easyocr
!git clone --depth 1 https://github.com/patrisiyarum/fire.git /content/fire 2>/dev/null || echo "Repo already cloned."
%cd /content/fire/flashinspector-ai

from pathlib import Path
print("\nProject ready at:", Path.cwd())
print("Model exists:", Path("best.pt").exists())

## 3. Upload your models

Upload both models from Kaggle:
- **`best_detect.pt`** — detection (S1: equipment, S2: marking signs, S5: inspection tags, S6: fire class symbols)
- **`best_segment.pt`** — segmentation (S3/S4: extinguisher condition/occlusion)

The notebook maps each detection to its **FireSafetyNet service** and:
- **S5 inspection tags** → crops the tag region and runs OCR to read expiry dates
- **S3/S4 condition masks** → flags blocked or non-compliant extinguishers
- **S6 fire class symbols** → identifies the fire classes on extinguisher labels

Upload both files at once, or upload just one — the notebook will use whichever models are available.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from google.colab import files

# --- Service classification for model classes ---
# Keywords that map model class names → FireSafetyNet services
SERVICE_KEYWORDS = {
    "S5-inspection_tag": ["tag", "inspection", "expir", "date", "maintenance", "servic"],
    "S6-fire_class": ["class", "symbol", "type_a", "type_b", "type_c", "type_d", "type_e", "type_f",
                       "class_a", "class_b", "class_c", "class_d", "class_e", "class_f",
                       "powder", "foam", "co2", "water", "wet_chemical"],
    "S2-marking": ["marking", "sign", "signage", "suppression"],
    "S1-equipment": ["extinguisher", "blanket", "detector", "call_point", "alarm", "sounder",
                     "exit", "dome", "flash", "light", "orb", "hydrant"],
}

# Single-letter fire class symbols: A=ordinary, B=liquids, C=electrical, D=metals, F=cooking
FIRE_CLASS_LETTERS = {"a", "b", "c", "d", "e", "f", "k"}

def classify_service(class_name):
    """Determine which FireSafetyNet service a class belongs to."""
    name_lower = class_name.lower().strip().replace(" ", "_").replace("-", "_")
    # Exact match: single-letter fire class symbols (A, B, C, D, E, F, K)
    if name_lower in FIRE_CLASS_LETTERS:
        return "S6-fire_class"
    # Numeric fire ratings from S6 dataset (e.g. "1", "13", "21B" → rating numbers)
    if name_lower.isdigit():
        return "S6-fire_class"
    for service, keywords in SERVICE_KEYWORDS.items():
        for kw in keywords:
            if kw in name_lower:
                return service
    return "unknown"

# Upload your models (select both best_detect.pt and best_segment.pt at once)
print("Upload your model files (best_detect.pt and/or best_segment.pt):")
print("You can select multiple files in the upload dialog.\n")

detect_model = None
segment_model = None

try:
    up = files.upload()
    uploaded_names = list(up.keys())
except Exception:
    uploaded_names = []

# Load uploaded models
for name in uploaded_names:
    p = Path(name)
    m = YOLO(str(p))
    t = m.overrides.get("task", "detect")
    if t == "segment":
        segment_model = m
        print(f"Segmentation model loaded: {p}")
        print(f"  Classes ({len(m.names)}):")
        for idx, cls in m.names.items():
            svc = classify_service(cls)
            print(f"    [{idx}] {cls}  → {svc}")
        print()
    else:
        detect_model = m
        print(f"Detection model loaded: {p}")
        print(f"  Classes ({len(m.names)}):")
        for idx, cls in m.names.items():
            svc = classify_service(cls)
            print(f"    [{idx}] {cls}  → {svc}")
        print()

# Fallback to repo best.pt if nothing uploaded
if detect_model is None and segment_model is None:
    fallback = Path("best.pt")
    if fallback.exists():
        detect_model = YOLO(str(fallback))
        print(f"Using fallback model from repo: {fallback}")
        print(f"  Classes ({len(detect_model.names)}):")
        for idx, cls in detect_model.names.items():
            svc = classify_service(cls)
            print(f"    [{idx}] {cls}  → {svc}")
        print()
    else:
        raise FileNotFoundError("No models found. Upload best_detect.pt and/or best_segment.pt.")

# Build service map from the loaded model(s)
detect_service_map = {}
if detect_model:
    for idx, cls in detect_model.names.items():
        detect_service_map[cls] = classify_service(cls)
seg_service_map = {}
if segment_model:
    for idx, cls in segment_model.names.items():
        seg_service_map[cls] = classify_service(cls)

# Summary
models_loaded = []
if detect_model:
    models_loaded.append("Detection")
if segment_model:
    models_loaded.append("Segmentation")
print(f"Models ready: {', '.join(models_loaded)}")

# Check for service coverage
has_s5 = any("S5" in v for v in detect_service_map.values())
has_s6 = any("S6" in v for v in detect_service_map.values())
has_s3s4 = any("S3" in v or "S4" in v for v in seg_service_map.values()) if segment_model else False
if has_s5:
    print("  S5 (inspection tags): will OCR detected tag regions for expiry dates")
if has_s6:
    print("  S6 (fire class symbols): will identify fire class symbols")
if has_s3s4:
    print("  S3/S4 (condition check): will flag blocked/non-compliant extinguishers")
if not has_s5 and not has_s6:
    print("\n  NOTE: No S5/S6 classes detected in model. If your model should detect")
    print("  inspection tags or fire class symbols, check the class names above.")
    print("  The model may use different naming — update SERVICE_KEYWORDS if needed.")

## 4. Upload your video

In [ ]:
from google.colab import files
from pathlib import Path

print("Select a video file (.mp4, .avi, .mov, .mkv):")
uploaded = files.upload()
VIDEO_PATH = Path(list(uploaded.keys())[0])
print(f"\nUploaded: {VIDEO_PATH} ({VIDEO_PATH.stat().st_size / 1024 / 1024:.1f} MB)")

## 5. Run inference on video (both models)

Runs both detection and segmentation models on each frame and combines the annotations.
- **CONFIDENCE**: minimum detection confidence (0.0–1.0). Lower values detect more (including small tags).
- **FRAME_SKIP**: process every Nth frame (higher = faster, lower = more detailed)

**Service-aware processing:**
- **S1** (equipment): fire extinguishers, blankets, smoke detectors, call points → bounding boxes
- **S2** (marking): FSE marking signs → bounding boxes
- **S3/S4** (condition): segmentation masks → flags blocked/non-compliant extinguishers
- **S5** (inspection tags): detects tag region → **OCR reads the date** → checks if expired
- **S6** (fire class symbols): detects symbols → identifies fire classes on extinguisher labels

In [ ]:
import cv2
import numpy as np
import re
import time
from datetime import datetime, date
from pathlib import Path

# ── Settings ──────────────────────────────────────────────────────────
CONFIDENCE = 0.15       # lower than default to catch small tags/symbols
FRAME_SKIP = 5          # process every Nth frame
OCR_CONF_MIN = 0.40     # only OCR extinguisher labels above this confidence
OCR_COOLDOWN = 10       # skip OCR for same-region detections within N frames

# ── Class name consolidation ──────────────────────────────────────────
CLASS_MAP = {
    "right exit": "emergency_exit", "left exit": "emergency_exit",
    "Right Exit": "emergency_exit", "Left Exit": "emergency_exit",
    "Straight Exit": "emergency_exit", "straight exit": "emergency_exit",
    "Left-Right Exit": "emergency_exit", "left-right exit": "emergency_exit",
    "emergency exit": "emergency_exit", "Emergency Exit": "emergency_exit",
    "fire-extinguisher": "fire_extinguisher",
    "fire extinguisher": "fire_extinguisher",
    "Fire_Extinguisher": "fire_extinguisher",
}

def consolidate_class(name):
    return CLASS_MAP.get(name, name)

# ── OCR setup for inspection tag reading (S5) ────────────────────────
import easyocr
ocr_reader = easyocr.Reader(["en"], gpu=True, verbose=False)
print("OCR reader loaded for inspection tag reading.\n")

def preprocess_for_ocr(crop):
    """Enhance a cropped region for better OCR accuracy."""
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    if max(h, w) < 200:
        scale = 200 / max(h, w)
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 15, 4)
    return binary

def is_useful_text(text):
    """Filter out garbled OCR noise — keep text that looks meaningful."""
    t = text.strip()
    if len(t) < 2:
        return False
    alnum = sum(c.isalnum() for c in t)
    if alnum == 0:
        return False
    if alnum / len(t) < 0.5:
        return False
    has_vowel_or_digit = bool(re.search(r'[aeiouAEIOU0-9]', t))
    if not has_vowel_or_digit and len(t) > 3:
        return False
    return True

def bbox_key(bbox, grid=80):
    """Quantize a bbox center to a grid cell for cooldown tracking."""
    cx = int((bbox[0] + bbox[2]) / 2) // grid
    cy = int((bbox[1] + bbox[3]) / 2) // grid
    return (cx, cy)

def read_tag_region(frame, bbox, padding=5):
    """Crop a detected tag/label region, preprocess, and OCR it."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1, y1 = max(0, x1 - padding), max(0, y1 - padding)
    x2, y2 = min(w, x2 + padding), min(h, y2 + padding)
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return []
    processed = preprocess_for_ocr(crop)
    texts = ocr_reader.readtext(processed, detail=0, paragraph=False)
    return [t for t in texts if is_useful_text(t)]

def extract_label_region(frame, bbox):
    """Extract the label area of an extinguisher (center band, 30-80% height).

    Fire extinguisher labels are typically in the middle portion of the body,
    not at the top (handle/nozzle) or bottom (base). This crops just the
    label band for much cleaner OCR.
    """
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    box_h = y2 - y1
    box_w = x2 - x1
    label_y1 = y1 + int(box_h * 0.30)
    label_y2 = y1 + int(box_h * 0.80)
    label_x1 = x1 + int(box_w * 0.10)
    label_x2 = x2 - int(box_w * 0.10)
    crop = frame[label_y1:label_y2, label_x1:label_x2]
    if crop.size == 0:
        return []
    processed = preprocess_for_ocr(crop)
    texts = ocr_reader.readtext(processed, detail=0, paragraph=False)
    return [t for t in texts if is_useful_text(t)]

def parse_dates(texts):
    """Try to extract dates from OCR text lines. Returns list of (date_obj, raw_text)."""
    found = []
    date_patterns = [
        (r'(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})', "dmy4"),
        (r'(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{2})', "dmy2"),
        (r'(\d{4})[/\-.](\d{1,2})[/\-.](\d{1,2})', "ymd"),
        (r'(\w+)\s+(\d{4})', "month_year"),
        (r'(\d{1,2})[/\-.](\d{4})', "my"),
    ]
    for text in texts:
        for pattern, fmt in date_patterns:
            m = re.search(pattern, text)
            if m:
                try:
                    if fmt == "dmy4":
                        d = datetime.strptime(f"{m.group(1)}/{m.group(2)}/{m.group(3)}", "%d/%m/%Y").date()
                    elif fmt == "dmy2":
                        d = datetime.strptime(f"{m.group(1)}/{m.group(2)}/{m.group(3)}", "%d/%m/%y").date()
                    elif fmt == "ymd":
                        d = datetime.strptime(f"{m.group(1)}/{m.group(2)}/{m.group(3)}", "%Y/%m/%d").date()
                    elif fmt == "month_year":
                        for mfmt in ["%B %Y", "%b %Y"]:
                            try:
                                d = datetime.strptime(f"{m.group(1)} {m.group(2)}", mfmt).date()
                                break
                            except ValueError:
                                continue
                        else:
                            continue
                    elif fmt == "my":
                        d = datetime.strptime(f"{m.group(1)}/{m.group(2)}", "%m/%Y").date()
                    found.append((d, text.strip()))
                except ValueError:
                    continue
    return found

def check_expiry(dates):
    """Check if any parsed dates are expired. Returns (is_expired, latest_date, raw_text)."""
    if not dates:
        return None, None, None
    today = date.today()
    latest = max(dates, key=lambda x: x[0])
    is_expired = latest[0] < today
    return is_expired, latest[0], latest[1]

# ── Service-aware color scheme ────────────────────────────────────────
SERVICE_COLORS = {
    "S1-equipment": (0, 255, 0),      # green
    "S2-marking": (255, 165, 0),       # orange
    "S5-inspection_tag": (0, 255, 255),# yellow
    "S6-fire_class": (255, 0, 255),    # magenta
    "unknown": (200, 200, 200),        # gray
}

# ── Open video ────────────────────────────────────────────────────────
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video: {VIDEO_PATH.name}")
print(f"Resolution: {width}x{height}, FPS: {fps:.1f}, Total frames: {total_frames}")
print(f"Duration: {total_frames / fps:.1f}s")
active = []
if detect_model: active.append("Detection")
if segment_model: active.append("Segmentation")
print(f"Running: {' + '.join(active)}  (conf={CONFIDENCE})")
print(f"OCR: label conf>={OCR_CONF_MIN}, cooldown={OCR_COOLDOWN} frames")
print(f"Processing every {FRAME_SKIP} frame(s)...\n")

# ── Output video ──────────────────────────────────────────────────────
out_dir = Path("inference_results")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / f"result_{VIDEO_PATH.stem}.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(str(out_path), fourcc, fps / FRAME_SKIP, (width, height))

# ── Process frames ────────────────────────────────────────────────────
detect_detections = []
segment_detections = []
tag_readings = []       # S5: OCR results from inspection tags
expiry_results = []     # S5: parsed expiry dates
symbol_detections = []  # S6: fire class symbols found
ocr_last_frame = {}     # cooldown tracker: bbox_key → last frame OCR'd
ocr_skipped = 0         # count of OCR calls skipped by cooldown

frame_idx = 0
processed = 0
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % FRAME_SKIP == 0:
        annotated = frame.copy()
        timestamp = round(frame_idx / fps, 2)

        # ── Segmentation model (condition/occlusion masks) ───────────
        if segment_model:
            seg_results = segment_model(frame, conf=CONFIDENCE, verbose=False)[0]
            annotated = seg_results.plot(img=annotated, boxes=False)
            for box in seg_results.boxes:
                raw_cls = seg_results.names[int(box.cls)]
                cls_name = consolidate_class(raw_cls)
                svc = seg_service_map.get(raw_cls, classify_service(raw_cls))
                segment_detections.append({
                    "frame": frame_idx, "timestamp_sec": timestamp,
                    "class": cls_name, "confidence": round(float(box.conf), 3),
                    "model": "segment", "service": svc,
                })

        # ── Detection model (S1, S2, S5, S6) ─────────────────────────
        if detect_model:
            det_results = detect_model(frame, conf=CONFIDENCE, verbose=False)[0]

            for box in det_results.boxes:
                bbox_float = [float(x) for x in box.xyxy[0].tolist()]
                bbox = [int(x) for x in box.xyxy[0].tolist()]
                raw_cls = det_results.names[int(box.cls)]
                cls_name = consolidate_class(raw_cls)
                conf_val = float(box.conf)
                svc = detect_service_map.get(raw_cls, classify_service(raw_cls))

                # Color by service
                color = SERVICE_COLORS.get(svc, SERVICE_COLORS["unknown"])

                # Draw bounding box
                cv2.rectangle(annotated, (bbox[0], bbox[1]), (bbox[2], bbox[3]), color, 2)
                label = f"{cls_name} {conf_val:.2f}"
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
                cv2.rectangle(annotated, (bbox[0], bbox[1] - th - 6), (bbox[0] + tw, bbox[1]), color, -1)
                cv2.putText(annotated, label, (bbox[0], bbox[1] - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)

                detect_detections.append({
                    "frame": frame_idx, "timestamp_sec": timestamp,
                    "class": cls_name, "confidence": round(conf_val, 3),
                    "model": "detect", "bbox": bbox_float, "service": svc,
                })

                # ── S5: Inspection tag → OCR for expiry date ─────────
                if "S5" in svc:
                    bk = bbox_key(bbox_float)
                    if frame_idx - ocr_last_frame.get(("s5", bk), -999) >= OCR_COOLDOWN:
                        ocr_last_frame[("s5", bk)] = frame_idx
                        texts = read_tag_region(frame, bbox_float)
                        if texts:
                            tag_readings.append({
                                "frame": frame_idx, "timestamp_sec": timestamp,
                                "texts": texts, "bbox": bbox_float,
                            })
                            dates = parse_dates(texts)
                            is_expired, exp_date, raw = check_expiry(dates)
                            if is_expired is not None:
                                expiry_results.append({
                                    "frame": frame_idx, "timestamp_sec": timestamp,
                                    "expired": is_expired, "date": exp_date, "raw": raw,
                                })
                                status_color = (0, 0, 255) if is_expired else (0, 200, 0)
                                status_text = f"EXPIRED {exp_date}" if is_expired else f"Valid until {exp_date}"
                                cv2.putText(annotated, status_text, (bbox[0], bbox[3] + 20),
                                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, status_color, 2, cv2.LINE_AA)
                            else:
                                y_off = bbox[3] + 18
                                for txt in texts[:3]:
                                    cv2.putText(annotated, txt[:50], (bbox[0], y_off),
                                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1, cv2.LINE_AA)
                                    y_off += 16
                    else:
                        ocr_skipped += 1

                # ── S6: Fire class symbol ─────────────────────────────
                if "S6" in svc:
                    symbol_detections.append({
                        "frame": frame_idx, "timestamp_sec": timestamp,
                        "class": cls_name, "confidence": round(conf_val, 3),
                    })
                    bk = bbox_key(bbox_float)
                    if frame_idx - ocr_last_frame.get(("s6", bk), -999) >= OCR_COOLDOWN:
                        ocr_last_frame[("s6", bk)] = frame_idx
                        sym_texts = read_tag_region(frame, bbox_float)
                        if sym_texts:
                            y_off = bbox[3] + 18
                            for txt in sym_texts[:2]:
                                cv2.putText(annotated, txt[:40], (bbox[0], y_off),
                                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 255), 1, cv2.LINE_AA)
                                y_off += 16
                    else:
                        ocr_skipped += 1

                # ── S1 equipment: OCR the label region of extinguishers
                if "extinguisher" in cls_name.lower() and "S1" in svc:
                    if conf_val >= OCR_CONF_MIN:
                        bk = bbox_key(bbox_float)
                        if frame_idx - ocr_last_frame.get(("ext", bk), -999) >= OCR_COOLDOWN:
                            ocr_last_frame[("ext", bk)] = frame_idx
                            texts = extract_label_region(frame, bbox_float)
                            if texts:
                                tag_readings.append({
                                    "frame": frame_idx, "timestamp_sec": timestamp,
                                    "texts": texts, "bbox": bbox_float, "source": "extinguisher_label",
                                })
                                dates = parse_dates(texts)
                                is_expired, exp_date, raw = check_expiry(dates)
                                if is_expired is not None:
                                    status_color = (0, 0, 255) if is_expired else (0, 200, 0)
                                    status_text = f"EXPIRED {exp_date}" if is_expired else f"Valid {exp_date}"
                                    cv2.putText(annotated, status_text, (bbox[0], bbox[3] + 20),
                                                cv2.FONT_HERSHEY_SIMPLEX, 0.55, status_color, 2, cv2.LINE_AA)
                                    expiry_results.append({
                                        "frame": frame_idx, "timestamp_sec": timestamp,
                                        "expired": is_expired, "date": exp_date, "raw": raw,
                                    })
                        else:
                            ocr_skipped += 1

        out_writer.write(annotated)
        processed += 1

        if processed % 50 == 0:
            elapsed = time.time() - start_time
            pct = frame_idx / total_frames * 100
            print(f"  {pct:.0f}% done — {processed} frames ({processed / elapsed:.1f} fps)")

    frame_idx += 1

cap.release()
out_writer.release()

elapsed = time.time() - start_time
all_detections = detect_detections + segment_detections
print(f"\nDone! Processed {processed} frames in {elapsed:.1f}s ({processed / max(elapsed, 0.01):.1f} fps)")
print(f"Annotated video saved: {out_path}")
if tag_readings:
    print(f"OCR: Read {len(tag_readings)} tag/label regions ({ocr_skipped} duplicate reads skipped by cooldown)")
if expiry_results:
    n_expired = sum(1 for e in expiry_results if e["expired"])
    print(f"Expiry check: {n_expired} expired, {len(expiry_results) - n_expired} valid")

## 6. Results — service-by-service breakdown + compliance report

Shows detections grouped by FireSafetyNet service, OCR readings from inspection tags, expiry status, and condition check results.

In [ ]:
from collections import Counter

def section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

# ── Detection model by service ────────────────────────────────────────
if detect_detections:
    section("DETECTION MODEL — by service")
    by_svc = {}
    for d in detect_detections:
        svc = d.get("service", "unknown")
        by_svc.setdefault(svc, []).append(d)
    for svc in sorted(by_svc):
        dets = by_svc[svc]
        counts = Counter(d["class"] for d in dets)
        print(f"\n  [{svc}] — {len(dets)} detections")
        for cls, count in counts.most_common():
            confs = [d["confidence"] for d in dets if d["class"] == cls]
            print(f"    {cls}: {count} (avg conf: {sum(confs)/len(confs):.2f})")

# ── Segmentation model ───────────────────────────────────────────────
if segment_detections:
    section("SEGMENTATION MODEL — mask detections")
    counts = Counter(d["class"] for d in segment_detections)
    print(f"  Total: {len(segment_detections)} mask detections\n")
    for cls, count in counts.most_common():
        confs = [d["confidence"] for d in segment_detections if d["class"] == cls]
        print(f"    {cls}: {count} (avg conf: {sum(confs)/len(confs):.2f})")
    seg_frames = len(set(d["frame"] for d in segment_detections))
    print(f"\n  Masks drawn on {seg_frames} frames")

    seg_services = set(d.get("service", "") for d in segment_detections)
    has_condition_classes = any("S3" in s or "S4" in s for s in seg_services)
    if has_condition_classes:
        cond_dets = [d for d in segment_detections if "S3" in d.get("service","") or "S4" in d.get("service","")]
        section("S3/S4 — CONDITION CHECK FLAGS")
        cond_counts = Counter(d["class"] for d in cond_dets)
        for cls, count in cond_counts.most_common():
            print(f"    {cls}: {count} detections")
        cond_frames = len(set(d["frame"] for d in cond_dets))
        print(f"\n  Condition issues flagged in {cond_frames} frames")
    else:
        print(f"\n  NOTE: Segmentation model classes map to general equipment, not")
        print(f"  specific S3/S4 condition categories. The masks show extinguisher")
        print(f"  locations — for condition/occlusion flagging, train with S3/S4 classes")
        print(f"  (e.g. 'blocked', 'damaged', 'obstructed', 'non_compliant').")

# ── S5: Inspection tag OCR readings (deduplicated by frequency) ──────
section("S5 — INSPECTION TAG READINGS (OCR)")
if tag_readings:
    # Count how many times each unique text was read across all frames
    text_counts = Counter()
    text_first_seen = {}  # text → first timestamp
    for tr in tag_readings:
        src = tr.get("source", "inspection_tag")
        for txt in tr["texts"]:
            norm = txt.strip()
            if norm:
                text_counts[(norm, src)] += 1
                if (norm, src) not in text_first_seen:
                    text_first_seen[(norm, src)] = tr["timestamp_sec"]

    print(f"  Read text from {len(tag_readings)} tag/label regions")
    print(f"  Unique text strings: {len(text_counts)}\n")

    # Show most frequently read text first (more likely to be real)
    print("  Most consistent reads (sorted by frequency):\n")
    for (txt, src), count in text_counts.most_common(30):
        first_t = text_first_seen[(txt, src)]
        freq_bar = "#" * min(count, 20)
        print(f"    {count:>3}x  ({src}) \"{txt}\"  [first @ {first_t}s]  {freq_bar}")
else:
    print("  No inspection tag text detected.")
    print("  Possible reasons:")
    print("    - Model has no S5 (inspection tag) class — check class list above")
    print("    - Tags too small or blurry in video — try lower CONFIDENCE or higher resolution")
    print("    - Model wasn't trained on S5 data")

# ── S5: Expiry date check ────────────────────────────────────────────
section("S5 — EXPIRY DATE CHECK")
if expiry_results:
    from datetime import date
    today = date.today()
    for e in expiry_results:
        status = "EXPIRED" if e["expired"] else "VALID"
        icon = "[X]" if e["expired"] else "[OK]"
        print(f"  {icon} {status}: date={e['date']} (from \"{e['raw']}\") @ {e['timestamp_sec']}s")
    n_expired = sum(1 for e in expiry_results if e["expired"])
    n_valid = len(expiry_results) - n_expired
    print(f"\n  Summary: {n_expired} EXPIRED, {n_valid} VALID (checked against today: {today})")
else:
    print("  No dates found on tags/labels.")
    print("  OCR could not parse any date patterns from the detected regions.")

# ── S6: Fire class symbols ───────────────────────────────────────────
if symbol_detections:
    section("S6 — FIRE CLASS SYMBOLS")
    sym_counts = Counter(d["class"] for d in symbol_detections)
    for cls, count in sym_counts.most_common():
        print(f"    {cls}: detected {count} times")

# ── Combined totals ──────────────────────────────────────────────────
section("COMBINED TOTALS")
print(f"  Detection model: {len(detect_detections)} detections")
print(f"  Segmentation model: {len(segment_detections)} mask detections")
print(f"  Tag/label OCR reads: {len(tag_readings)}")
print(f"  Expiry dates found: {len(expiry_results)}")
print(f"  Fire class symbols: {len(symbol_detections)}")

if not detect_detections and not segment_detections:
    print(f"\n  No detections found. Try lowering CONFIDENCE (currently {CONFIDENCE}).")

## 7. Preview a sample frame

In [ ]:
import cv2
from IPython.display import display, Image as IPImage
from pathlib import Path
import tempfile

# Open the annotated video and grab a frame from the middle
cap = cv2.VideoCapture(str(out_path))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)  # jump to middle
ret, frame = cap.read()
cap.release()

if ret:
    tmp = Path(tempfile.mktemp(suffix=".jpg"))
    cv2.imwrite(str(tmp), frame)
    display(IPImage(str(tmp), width=800))
    print(f"Showing frame {total // 2} of {total}")
else:
    print("Could not read a frame from the output video.")

## 8. Download the annotated video

In [ ]:
from google.colab import files
from pathlib import Path

if Path(out_path).exists():
    print(f"Downloading: {out_path} ({Path(out_path).stat().st_size / 1024 / 1024:.1f} MB)")
    files.download(str(out_path))
else:
    print("Output video not found. Run the inference cell above first.")